# string

Strings are the backbone of logs, API payloads, CLI arguments, file paths, config values, and text-based protocols. Many production bugs come from assumptions about encoding, whitespace, splitting, and immutability.


## 1. Problem First

Why does this matter in real systems?

- Logs arrive as strings before becoming structured data.
- API contracts often encode booleans and numbers as text.
- Tiny whitespace bugs break parsing and auth.


In [ ]:
line = "ERROR|db timeout|service-a"
parts = line.split("|")
print(parts)


## 2. Minimal Concept

Strings are sequences of Unicode characters.

- Create with quotes.
- Index and slice like sequences.
- Use `.strip()`, `.split()`, `.replace()`, `.lower()`.


## 3. Mental Model

How Python actually behaves

Strings are immutable objects. Any apparent modification creates a new string. Visible character count and encoded byte count can differ, which matters for files, APIs, and storage.


In [ ]:
message = "api"
print(id(message))
message = message.upper()
print(id(message))

text = "cafe"
print(len(text), len(text.encode("utf-8")))


## 4. Internal Mechanics

Concatenation creates new strings. Methods return new values instead of mutating in place. Encoding converts strings to bytes, which is where files and protocols actually operate.


In [ ]:
import dis

def normalize(value):
    return value.strip().lower()

print(id("api"), id("api".strip()))
dis.dis(normalize)


## 5. Edge Cases

Where it breaks

- `.split(",")` behaves differently from `.split()`.
- Hidden whitespace breaks equality.
- Case mismatches break comparisons.
- Repeated concatenation in loops can be expensive.


In [ ]:
print("a  b".split())
print("a,,b".split(","))

token = " admin \n"
print(token == "admin")
print(token.strip() == "admin")


## 6. Performance Thinking

- Indexing is O(1).
- Slicing is O(n) in slice length.
- Repeated concatenation can become O(n^2).
- `''.join(parts)` is better for many fragments.


## 7. Real Use Case

A CLI tool reads raw log lines and normalizes them before shipping structured records.


In [ ]:
raw_line = "  WARN|Disk almost full|node-1  "
level, message, host = raw_line.strip().split("|")
record = {
    "level": level,
    "message": message,
    "host": host,
}
print(record)


## 8. Anti-Pattern

What beginners do wrong

- Forget to normalize whitespace.
- Concatenate in large loops without thinking about cost.
- Assume all text is clean ASCII.


In [ ]:
result = ""
for part in ["a", "b", "c"]:
    result += part
print(result)

better = "".join(["a", "b", "c"])
print(better)


## 9. Interview Signals

Questions FAANG asks

- Why are Python strings immutable?
- Why is `join()` preferred over repeated concatenation?
- What is the difference between characters and encoded bytes?
- How would you normalize untrusted text input safely?


## 10. Exercise (Non-trivial)

Design a parser for inconsistent log lines that handles extra spaces, missing optional fields, and mixed-case severity without corrupting the output schema.


In [ ]:
def parse_log_line(line):
    parts = line.strip().split("|")
    return parts

# Task:
# 1. Normalize spacing and severity case.
# 2. Handle missing optional fields.
# 3. Return a stable dictionary schema.
